## Dataset Feature Inventory

Inspect the processed customer dataset before selecting features for customer segmentation.

In [1]:
import pandas as pd
from pathlib import Path

processed_dataset_path = Path(
    "../data/processed/processed_dataset.csv"
)

processed_dataset = pd.read_csv(
    processed_dataset_path
)

print(f"Dataset Shape: {processed_dataset.shape}")

print("\nColumns:")
print(processed_dataset.columns.tolist())

print("\nData Types:")
print(processed_dataset.dtypes)

Dataset Shape: (96096, 5)

Columns:
['customer_unique_id', 'total_orders', 'total_spent', 'average_order_value', 'average_review_score']

Data Types:
customer_unique_id          str
total_orders              int64
total_spent             float64
average_order_value     float64
average_review_score    float64
dtype: object


## Segmentation Feature Selection

Select customer behavior, spending, and satisfaction features for clustering while excluding the customer identifier.

In [2]:
segmentation_features = [
    "total_orders",
    "total_spent",
    "average_order_value",
    "average_review_score"
]

customer_segmentation_data = processed_dataset[
    segmentation_features
].copy()

print("Selected Features:")
print(segmentation_features)

print(
    f"\nSegmentation Data Shape: "
    f"{customer_segmentation_data.shape}"
)

print("\nSelected Feature Data Types:")
print(customer_segmentation_data.dtypes)

Selected Features:
['total_orders', 'total_spent', 'average_order_value', 'average_review_score']

Segmentation Data Shape: (96096, 4)

Selected Feature Data Types:
total_orders              int64
total_spent             float64
average_order_value     float64
average_review_score    float64
dtype: object


## Clustering Preprocessing

Prepare the selected segmentation features for clustering by handling missing values and standardizing their numerical scale.

In [5]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Handle missing values using feature-wise median imputation
imputer = SimpleImputer(strategy="median")

imputed_segmentation_data = imputer.fit_transform(
    customer_segmentation_data
)

# Standardize features for distance-based clustering
scaler = StandardScaler()

scaled_segmentation_data = scaler.fit_transform(
    imputed_segmentation_data
)

print(
    f"Imputed Data Shape: "
    f"{imputed_segmentation_data.shape}"
)

print(
    f"Scaled Data Shape: "
    f"{scaled_segmentation_data.shape}"
)

print(
    f"Remaining NaN Values: "
    f"{pd.isna(scaled_segmentation_data).sum()}"
)

Imputed Data Shape: (96096, 4)
Scaled Data Shape: (96096, 4)
Remaining NaN Values: 0


## K-Means Cluster Selection

Evaluate multiple cluster counts using inertia and silhouette score to identify an appropriate number of customer segments.

In [6]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import pandas as pd

cluster_results = []

# Use a representative sample for efficient model selection
sample_size = min(10000, len(scaled_segmentation_data))

sample_data = pd.DataFrame(
    scaled_segmentation_data
).sample(
    n=sample_size,
    random_state=42
).values

for k in range(2, 9):

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(sample_data)

    inertia = kmeans.inertia_

    silhouette = silhouette_score(
        sample_data,
        labels
    )

    cluster_results.append({
        "n_clusters": k,
        "inertia": inertia,
        "silhouette_score": silhouette
    })

cluster_results = pd.DataFrame(
    cluster_results
)

print(cluster_results)

   n_clusters       inertia  silhouette_score
0           2  26919.203750          0.554998
1           3  19441.455847          0.600887
2           4  12735.856054          0.620453
3           5   9727.333390          0.626089
4           6   7831.929812          0.632181
5           7   6537.527685          0.558541
6           8   5723.391689          0.575461


## Clustering Algorithm Comparison

Compare multiple clustering algorithms using consistent evaluation metrics before selecting the final segmentation model.

In [7]:
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
import pandas as pd

# Use a fixed representative sample for efficient model comparison
sample_size = min(10000, len(scaled_segmentation_data))

model_selection_sample = pd.DataFrame(
    scaled_segmentation_data
).sample(
    n=sample_size,
    random_state=42
).values

comparison_results = []

models = {
    "K-Means": KMeans(
        n_clusters=6,
        random_state=42,
        n_init=10
    ),
    "MiniBatch K-Means": MiniBatchKMeans(
        n_clusters=6,
        random_state=42,
        n_init=10,
        batch_size=1024
    ),
    "Gaussian Mixture": GaussianMixture(
        n_components=6,
        random_state=42
    )
}

for model_name, model in models.items():

    model.fit(model_selection_sample)

    labels = model.predict(model_selection_sample)

    silhouette = silhouette_score(
        model_selection_sample,
        labels
    )

    davies_bouldin = davies_bouldin_score(
        model_selection_sample,
        labels
    )

    comparison_results.append({
        "Model": model_name,
        "Silhouette Score": silhouette,
        "Davies-Bouldin Index": davies_bouldin
    })

model_comparison = pd.DataFrame(
    comparison_results
)

print(model_comparison)

               Model  Silhouette Score  Davies-Bouldin Index
0            K-Means          0.632181              0.655880
1  MiniBatch K-Means          0.481821              0.923802
2   Gaussian Mixture          0.173988              2.542098


## Final Training Data

Prepare the complete customer-level feature matrix using the locked segmentation features and preserve customer identifiers separately.

In [3]:
from pathlib import Path
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Load the final processed dataset
processed_dataset_path = Path(
    "../data/processed/processed_dataset.csv"
)

processed_dataset = pd.read_csv(
    processed_dataset_path
)

# Locked segmentation feature set
segmentation_features = [
    "total_orders",
    "total_spent",
    "average_order_value",
    "average_review_score"
]

# Preserve customer identifiers separately
customer_ids = processed_dataset[
    "customer_unique_id"
].copy()

# Select model features
training_features = processed_dataset[
    segmentation_features
].copy()

# Handle missing values
final_imputer = SimpleImputer(
    strategy="median"
)

imputed_training_features = final_imputer.fit_transform(
    training_features
)

# Standardize features for distance-based clustering
final_scaler = StandardScaler()

scaled_training_features = final_scaler.fit_transform(
    imputed_training_features
)

print(f"Customer Records: {len(customer_ids)}")
print(
    f"Training Feature Shape: "
    f"{training_features.shape}"
)
print(
    f"Scaled Feature Shape: "
    f"{scaled_training_features.shape}"
)

print(
    f"\nRemaining NaN Values: "
    f"{pd.isna(scaled_training_features).sum()}"
)

print("\nTraining Features:")
print(training_features.columns.tolist())

Customer Records: 96096
Training Feature Shape: (96096, 4)
Scaled Feature Shape: (96096, 4)

Remaining NaN Values: 0

Training Features:
['total_orders', 'total_spent', 'average_order_value', 'average_review_score']


## Train Customer Segmentation Model

Train the final K-Means segmentation pipeline using the complete customer-level feature dataset and the selected six-cluster configuration.

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Final Customer Segmentation pipeline
segmentation_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            KMeans(
                n_clusters=6,
                random_state=42,
                n_init=10
            )
        )
    ]
)

# Train on the complete customer dataset
segmentation_pipeline.fit(
    training_features
)

# Generate cluster assignments
cluster_labels = segmentation_pipeline.predict(
    training_features
)

print("Customer Segmentation Model trained successfully.")

print(
    f"\nTotal Customers: "
    f"{len(cluster_labels)}"
)

print(
    f"Number of Clusters: "
    f"{segmentation_pipeline.named_steps['model'].n_clusters}"
)

print("\nCluster Distribution:")
print(
    pd.Series(
        cluster_labels,
        name="cluster"
    ).value_counts().sort_index()
)

Customer Segmentation Model trained successfully.

Total Customers: 96096
Number of Clusters: 6

Cluster Distribution:
cluster
0    69743
1     2973
2    20488
3     2848
4       43
5        1
Name: count, dtype: int64
